# Advanced ANN for Churn Modelling
This notebook implements an Artificial Neural Network with:
1. **Cross-Validation** (via GridSearchCV)
2. **Grid Search** for Hyperparameter Tuning
3. **Model Checkpointing** to save the best version

### Prerequisites
Run the cell below to install SciKeras, which allows Keras to work with Scikit-Learn's Grid Search.

In [ ]:
!pip install scikeras

### 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Load the dataset
df = pd.read_csv('Churn_Modelling.csv')

# Features (X) and Label (y)
# We drop RowNumber, CustomerId, and Surname (indices 0, 1, 2)
X = df.iloc[:, 3:-1].values
y = df.iloc[:, -1].values

print("Data Loaded Successfully")

### 2. Data Preprocessing
We need to encode 'Gender' (Label Encoding) and 'Geography' (One Hot Encoding), then scale the features.

In [ ]:
# Encoding Categorical Data (Gender: Label Encoding)
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])

# Encoding Categorical Data (Geography: One Hot Encoding)
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

# Splitting dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

print("Preprocessing Complete")

### 3. Define Model and Grid Search
We define a function to build the ANN and use `GridSearchCV` for cross-validation and parameter optimization.

In [ ]:
def create_model(optimizer='adam', dropout_rate=0.1):
    model = Sequential([
        Dense(units=16, activation='relu', input_dim=X_train.shape[1]),
        Dropout(dropout_rate),
        Dense(units=8, activation='relu'),
        Dropout(dropout_rate),
        Dense(units=1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Wrap Keras for Scikit-Learn
model_wrapper = KerasClassifier(model=create_model, verbose=0)

# Hyperparameters to test
parameters = {
    'batch_size': [25, 32],
    'epochs': [50, 100],
    'optimizer': ['adam', 'rmsprop'],
    'model__dropout_rate': [0.1, 0.2]
}

grid_search = GridSearchCV(estimator=model_wrapper,
                           param_grid=parameters,
                           scoring='accuracy',
                           cv=5)

print("Starting Grid Search (this may take a few minutes)...")
grid_search = grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Accuracy: {grid_search.best_score_:.4f}")

### 4. Final Training with Model Checkpoint
We train the final model using the best parameters and save the best weights to a file.

In [ ]:
best_params = grid_search.best_params_

# Setup Checkpoint
checkpoint = ModelCheckpoint(
    filepath='best_churn_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Rebuild model with best params
final_model = create_model(optimizer=best_params['optimizer'], 
                           dropout_rate=best_params['model__dropout_rate'])

# Train
final_model.fit(
    X_train, y_train, 
    validation_data=(X_test, y_test), 
    batch_size=best_params['batch_size'], 
    epochs=best_params['epochs'], 
    callbacks=[checkpoint]
)